# 21. Controlled Output-Head Ablation Benchmark

This notebook benchmarks the five candidate prediction heads under an identical standardized ResNet-50 feature representation ($D = 512$):
1. **Softmax Cross-Entropy** (`Softmax_CE`)
2. **Softmax Continuous QWK** (`Softmax_QWK`)
3. **CORAL** (Consistent Rank Logits)
4. **CORN** (Conditional Rank Neurons)
5. **Cumulative Link Model + QWK** (`CLM_QWK`)

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

summary_json = Path("results/head_ablation/summary.json")
summary_csv = Path("results/head_ablation/summary.csv")

if summary_json.exists():
    with open(summary_json, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"Experiment: {data['experiment']} | Backbone: {data['backbone']}")
    df_summary = pd.DataFrame(data['heads']).T
    print(df_summary[['test_qwk', 'test_accuracy', 'within_1_accuracy', 'rdr_auc']])
else:
    print("Head ablation summary not yet generated. Run: python scripts/run_head_ablation.py --all-heads")

In [ ]:
# Inspect mathematical properties of the ordinal heads
import torch
from drdg.models.factory import build_ablation_model

for head in ["softmax_ce", "softmax_qwk", "coral", "corn", "clm_qwk"]:
    model = build_ablation_model(head, pretrained=False)
    x = torch.randn(2, 3, 512, 512)
    out = model(x)
    probs = out["probs"]
    print(f"Head: {head:12s} | Probs shape: {str(probs.shape):12s} | Normalized: {torch.allclose(probs.sum(-1), torch.ones(2), atol=1e-4)}")